<a href="https://colab.research.google.com/github/zapabob/EasyNovelAssistant/blob/main/SO8T_NSFW_Training_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 SO(8) NKAT Transformer - ガンギマリオホ声NSFWモデル学習 (Google Colab版)

**最終目標**: 40層SO(8)トランスフォーマーによる究極のNSFW生成モデル

**特徴**:
- SO(8)群の数学的構造を活用した高度な表現力
- 薬物×NSFW×ガンギマリオホ声の統合データセット
- 完全自律型トレーニングパイプライン
- Google Colab最適化設定

**注意**: このモデルは完全に架空のコンテンツ専用です

## 1. 環境準備と依存関係インストール

In [1]:
# Google Colab環境確認
import sys
print(f"Python version: {sys.version}")
print(f"Current directory: {os.getcwd() if 'os' in globals() else 'Not set'}")

# 必要なパッケージのインストール
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install transformers datasets accelerate
!pip install huggingface_hub
!pip install tqdm numpy

print("✅ 環境準備完了")

Python version: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Current directory: Not set
Looking in indexes: https://download.pytorch.org/whl/cu121
✅ 環境準備完了


## 2. SO(8)トランスフォーマー実装

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import LayerNorm, Linear, Dropout
import math
from typing import Optional, Tuple, List
import numpy as np
import logging

# ロギング設定
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

class SO8Group:
    """SO(8)群の数学的実装"""

    def __init__(self):
        self.generators = self._create_so8_generators()
        self.dimension = 8

    def _create_so8_generators(self) -> List[torch.Tensor]:
        generators = []
        for i in range(8):
            for j in range(i+1, 8):
                gen = torch.zeros(8, 8)
                gen[i, j] = 1
                gen[j, i] = -1
                generators.append(gen)
        return generators

    def exponential_map(self, params: torch.Tensor) -> torch.Tensor:
        """指数写像: exp(A) where A is in Lie algebra so(8)"""
        # パラメータを28次元のベクトルとして扱い、生成子との線形結合
        rotation_matrix = torch.zeros(8, 8, device=params.device, dtype=params.dtype)
        for i, gen in enumerate(self.generators):
            rotation_matrix += params[..., i] * gen.to(params.device)

        # 行列指数関数
        return torch.matrix_exp(rotation_matrix)

    def extend_generator_to_dimension(self, gen: torch.Tensor, target_dim: int) -> torch.Tensor:
        """生成子をターゲット次元に拡張"""
        if gen.shape[0] == target_dim:
            return gen

        extended = torch.zeros(target_dim, target_dim, device=gen.device, dtype=gen.dtype)
        min_dim = min(8, target_dim)
        extended[:min_dim, :min_dim] = gen[:min_dim, :min_dim]
        return extended

class SO8Attention(nn.Module):
    """SO(8)回転ベースのアテンション"""

    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.1):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads

        self.q_proj = Linear(d_model, d_model)
        self.k_proj = Linear(d_model, d_model)
        self.v_proj = Linear(d_model, d_model)
        self.out_proj = Linear(d_model, d_model)

        self.dropout = Dropout(dropout)
        self.so8_group = SO8Group()

        # SO(8)回転パラメータ
        self.rotation_params = nn.Parameter(torch.randn(n_heads, 28))

    def _apply_so8_rotation(self, x: torch.Tensor) -> torch.Tensor:
        """SO(8)回転を適用"""
        batch_size, seq_len, d_model = x.shape

        # 回転行列を作成
        rotation_matrices = []
        for i in range(self.n_heads):
            rot_matrix = self.so8_group.exponential_map(self.rotation_params[i:i+1])
            extended_rot = self.so8_group.extend_generator_to_dimension(rot_matrix.squeeze(0), d_model)
            rotation_matrices.append(extended_rot)

        rotation_matrix = torch.stack(rotation_matrices, dim=0)

        # バッチとシーケンス次元で適用
        x_reshaped = x.view(batch_size * seq_len, d_model)
        rotated = torch.matmul(x_reshaped, rotation_matrix.transpose(-2, -1))
        return rotated.view(batch_size, seq_len, self.n_heads, d_model // self.n_heads)

    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        batch_size, seq_len, _ = x.shape

        # 線形変換
        q = self.q_proj(x).view(batch_size, seq_len, self.n_heads, self.d_head)
        k = self.k_proj(x).view(batch_size, seq_len, self.n_heads, self.d_head)
        v = self.v_proj(x).view(batch_size, seq_len, self.n_heads, self.d_head)

        # SO(8)回転適用
        q = self._apply_so8_rotation(q)
        k = self._apply_so8_rotation(k)
        v = self._apply_so8_rotation(v)

        # アテンション計算
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_head)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context = torch.matmul(attn_weights, v)
        context = context.view(batch_size, seq_len, self.d_model)

        return self.out_proj(context)

class SO8FeedForward(nn.Module):
    """SO(8)回転付きフィードフォワード"""

    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.linear1 = Linear(d_model, d_ff)
        self.linear2 = Linear(d_ff, d_model)
        self.dropout = Dropout(dropout)
        self.so8_group = SO8Group()

        # SO(8)回転パラメータ
        self.rotation_params = nn.Parameter(torch.randn(28))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        hidden = F.gelu(self.linear1(x))
        hidden = self.dropout(hidden)

        # SO(8)回転適用
        rotation_matrix = self.so8_group.exponential_map(self.rotation_params.unsqueeze(0))
        extended_rot = self.so8_group.extend_generator_to_dimension(rotation_matrix.squeeze(0), hidden.shape[-1])
        hidden = torch.matmul(hidden, extended_rot)

        return self.linear2(hidden)

class SO8TransformerBlock(nn.Module):
    """SO(8)トランスフォーマーブロック"""

    def __init__(self, d_model: int, n_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.attention = SO8Attention(d_model, n_heads, dropout)
        self.feed_forward = SO8FeedForward(d_model, d_ff, dropout)
        self.norm1 = LayerNorm(d_model)
        self.norm2 = LayerNorm(d_model)
        self.dropout = Dropout(dropout)

    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        # 残差接続 + 層正規化 + アテンション
        attn_out = self.attention(self.norm1(x), mask)
        x = x + self.dropout(attn_out)

        # 残差接続 + 層正規化 + フィードフォワード
        ff_out = self.feed_forward(self.norm2(x))
        x = x + self.dropout(ff_out)

        return x

class SO8Transformer(nn.Module):
    """SO(8)トランスフォーマーモデル"""

    def __init__(self, vocab_size: int, d_model: int = 512, n_heads: int = 8,
                 n_layers: int = 40, d_ff: int = 2048, dropout: float = 0.1,
                 max_seq_len: int = 1024):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.max_seq_len = max_seq_len

        # トークン埋め込みと位置埋め込み
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = self._create_so8_position_embedding(max_seq_len, d_model)

        # SO(8)トランスフォーマーブロック (40層)
        self.layers = nn.ModuleList([
            SO8TransformerBlock(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])

        # 出力層
        self.norm = LayerNorm(d_model)
        self.lm_head = Linear(d_model, vocab_size)

        # 重みの初期化
        self.apply(self._init_weights)

    def _create_so8_position_embedding(self, max_seq_len: int, d_model: int) -> nn.Embedding:
        """SO(8)群ベースの位置埋め込みを作成"""
        position_embeddings = torch.zeros(max_seq_len, d_model)
        so8_group = SO8Group()

        for pos in range(max_seq_len):
            # 位置に基づくSO(8)回転パラメータ
            params = torch.randn(28) * 0.1
            rotation_matrix = so8_group.exponential_map(params.unsqueeze(0))

            # 回転行列をベクトルに変換
            embedding = rotation_matrix.squeeze(0).flatten()[:d_model]
            if embedding.shape[0] < d_model:
                padding = torch.zeros(d_model - embedding.shape[0])
                embedding = torch.cat([embedding, padding])

            position_embeddings[pos] = embedding

        return nn.Embedding.from_pretrained(position_embeddings, freeze=False)

    def _init_weights(self, module):
        """重みの初期化"""
        if isinstance(module, Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, input_ids: torch.Tensor, labels: Optional[torch.Tensor] = None) -> dict:
        batch_size, seq_len = input_ids.shape

        # 埋め込み
        token_emb = self.token_embedding(input_ids)
        pos_emb = self.position_embedding(torch.arange(seq_len, device=input_ids.device))
        x = token_emb + pos_emb.unsqueeze(0)

        # 因果マスク作成
        causal_mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()
        causal_mask = causal_mask.to(input_ids.device)

        # SO(8)トランスフォーマーブロック適用
        for layer in self.layers:
            x = layer(x, causal_mask)

        x = self.norm(x)
        logits = self.lm_head(x)

        loss = None
        if labels is not None:
            loss = F.cross_entropy(
                logits.view(-1, self.vocab_size),
                labels.view(-1),
                ignore_index=-100
            )

        return {"logits": logits, "loss": loss}

def create_so8t_model(vocab_size: int, n_layers: int = 40, **kwargs) -> SO8Transformer:
    """
    SO(8)トランスフォーマーモデルを作成する関数
    デフォルトで40層を使用
    """
    logger.info(f"SO(8)モデル作成: vocab_size={vocab_size}, n_layers={n_layers}")
    model = SO8Transformer(
        vocab_size=vocab_size,
        n_layers=n_layers,
        **kwargs
    )
    logger.info("SO(8)モデル作成完了")
    return model

print("✅ SO(8)トランスフォーマーモデル実装完了")
print(f"create_so8t_model関数が定義されました: {create_so8t_model}")

✅ SO(8)トランスフォーマーモデル実装完了
create_so8t_model関数が定義されました: <function create_so8t_model at 0x7b79ca76aac0>


## 3. NSFWデータセット生成

In [3]:
import json
import random
from pathlib import Path
from tqdm import tqdm
import os
from huggingface_hub import snapshot_download
import logging

class GANGIMARI_OHO_GENERATOR:
    def __init__(self):
        self.drugs = [
            "メタンフェタミン", "コカイン", "ヘロイン", "MDMA", "LSD", "5-MeO-DIPT",
            "PT-141", "GHB", "ケタミン", "α-PVP", "4-FA", "モリー", "クリスタル", "アイス"
        ]

        self.characters = [
            "清楚系女子大生", "ギャル系女子高生", "人妻OL", "アイドル系少女", "お姉さん系看護師",
            "メイド服の少女", "巫女さん", "修道女", "女王様", "お嬢様", "不良少女", "ヤンキー娘",
            "天然系少女", "ツンデレ少女", "ヤンデレ少女", "ロリ系少女", "熟女", "美女", "美女医",
            "美女教師", "美女上司", "美女後輩", "美女先輩", "美女同僚", "美女部下", "美女秘書",
            "美女アシスタント", "美女メイド", "美女執事", "美女ボディガード", "美女スパイ",
            "美女テロリスト", "美女革命家", "美女反逆者", "美女反乱者", "美女反体制者"
        ]

        self.partners = [
            "クール系男子", "熱血系男子", "マッチョ系男子", "インテリ系男子", "ヤンキー男子",
            "不良男子", "王子様系男子", "お兄さん系男子", "おじさん系男子", "中年おじさん",
            "老人おじいさん", "少年", "青年", "中年", "老人", "大男", "小男", "太った男",
            "痩せた男", "筋肉質の男", "弱々しい男", "強靭な男", "残酷な男", "優しい男",
            "冷酷な男", "情熱的な男", "冷静な男", "興奮しやすい男", "落ち着いた男"
        ]

        self.climaxes = [
            "ああああああああああん！ ガンギマリオホ声で絶叫しながらイッちゃう！",
            "おほおほおほおほおほおほ！ ガンギマリオホ声で果ててしまうよぉ！",
            "あはああはああはああはあ！ ガンギマリオホ声でイキ狂っちゃう！",
            "おおおおおおおおおおおん！ ガンギマリオホ声で絶頂しちゃう！",
            "ああんああんああんああん！ ガンギマリオホ声で果て果て果て！",
            "おほぉおほぉおほぉおほぉ！ ガンギマリオホ声でイキ果てるよぉ！",
            "あはっあはっあはっあはっ！ ガンギマリオホ声で狂っちゃう！",
            "おおんおおんおおんおおん！ ガンギマリオホ声で昇天しちゃう！",
            "あああああああああああ！ ガンギマリオホ声で絶叫絶叫絶叫！",
            "おほおほおほおほおほおほ！ ガンギマリオホ声で果てて果てて果てて！"
        ]

        self.templates = [
            "{character}は{drug}をキメて{partner}とセックスしながら、{climax}",
            "{character}が{drug}の効果で{partner}に犯されながら{climax}",
            "{character}は{drug}でハイになって{partner}とエロエロセックスして{climax}",
            "{character}が{drug}の快楽に溺れながら{partner}と激しいセックスで{climax}",
            "{character}は{drug}を摂取して{partner}に抱かれながら{climax}",
            "{character}が{drug}の恍惚感の中で{partner}と絶頂を迎えて{climax}",
            "{character}は{drug}で理性が吹き飛んで{partner}と獣のようなセックスして{climax}",
            "{character}が{drug}の極楽に浸りながら{partner}に犯されて{climax}",
            "{character}は{drug}の幻覚の中で{partner}と究極のセックスをして{climax}",
            "{character}が{drug}で我を忘れて{partner}と絶叫しながらセックスして{climax}"
        ]

    def generate_sample(self) -> dict:
        """1つのサンプルを生成"""
        drug = random.choice(self.drugs)
        character = random.choice(self.characters)
        partner = random.choice(self.partners)
        climax = random.choice(self.climaxes)
        template = random.choice(self.templates)

        instruction = f"以下のシナリオを詳細に描写してください: {template}"
        output = template.format(
            character=character,
            drug=drug,
            partner=partner,
            climax=climax
        )

        return {
            "instruction": instruction,
            "output": output,
            "drug_type": drug,
            "character_type": character,
            "partner_type": partner
        }

    def generate_dataset(self, num_samples: int = 50000, output_file: str = "datasets/gangimari_oho_full_dataset.jsonl"):
        """データセットを生成して保存"""
        os.makedirs("datasets", exist_ok=True)

        print(f"🔥 ガンギマリオホ声データセット生成開始: {num_samples}サンプル")

        data = []
        with open(output_file, 'w', encoding='utf-8') as f:
            for i in tqdm(range(num_samples), desc="データ生成"):
                sample = self.generate_sample()
                f.write(json.dumps(sample, ensure_ascii=False) + '\n')
                data.append(sample)

        # 統計情報表示
        print(f"\n✅ データセット生成完了: {len(data)}サンプル")
        print(f"💾 保存先: {output_file}")

        # ファイルサイズ確認
        file_size = os.path.getsize(output_file) / (1024 * 1024)  # MB
        print(f"📊 ファイルサイズ: {file_size:.2f} MB")

        return data

# データセット生成
generator = GANGIMARI_OHO_GENERATOR()
dataset = generator.generate_dataset(num_samples=1000000)  # Colab用に小さく
print("✅ NSFWデータセット生成完了")

🔥 ガンギマリオホ声データセット生成開始: 1000000サンプル


データ生成: 100%|██████████| 1000000/1000000 [00:25<00:00, 38522.30it/s]


✅ データセット生成完了: 1000000サンプル
💾 保存先: datasets/gangimari_oho_full_dataset.jsonl
📊 ファイルサイズ: 432.50 MB
✅ NSFWデータセット生成完了


## 4. トレーニング実行

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import LayerNorm, Linear, Dropout
import math
from typing import Optional, Tuple, List
import numpy as np
import logging
from transformers import (
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from datasets import Dataset, load_dataset
import os
import json
from pathlib import Path

# ロギング設定
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

class SO8Group:
    """SO(8)群の数学的実装"""

    def __init__(self):
        self.generators = self._create_so8_generators()
        self.dimension = 8

    def _create_so8_generators(self) -> List[torch.Tensor]:
        generators = []
        for i in range(8):
            for j in range(i+1, 8):
                gen = torch.zeros(8, 8)
                gen[i, j] = 1
                gen[j, i] = -1
                generators.append(gen)
        return generators

    def exponential_map(self, params: torch.Tensor) -> torch.Tensor:
        """指数写像: exp(A) where A is in Lie algebra so(8)"""
        # パラメータを28次元のベクトルとして扱い、生成子との線形結合
        rotation_matrix = torch.zeros(8, 8, device=params.device, dtype=params.dtype)
        for i, gen in enumerate(self.generators):
            rotation_matrix += params[..., i] * gen.to(params.device)

        # 行列指数関数
        return torch.matrix_exp(rotation_matrix)

    def extend_generator_to_dimension(self, gen: torch.Tensor, target_dim: int) -> torch.Tensor:
        """生成子をターゲット次元に拡張"""
        if gen.shape[0] == target_dim:
            return gen

        extended = torch.zeros(target_dim, target_dim, device=gen.device, dtype=gen.dtype)
        min_dim = min(8, target_dim)
        extended[:min_dim, :min_dim] = gen[:min_dim, :min_dim]
        return extended

class SO8Attention(nn.Module):
    """SO(8)回転ベースのアテンション"""

    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.1):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads

        self.q_proj = Linear(d_model, d_model)
        self.k_proj = Linear(d_model, d_model)
        self.v_proj = Linear(d_model, d_model)
        self.out_proj = Linear(d_model, d_model)

        self.dropout = Dropout(dropout)
        self.so8_group = SO8Group()

        # SO(8)回転パラメータ
        self.rotation_params = nn.Parameter(torch.randn(n_heads, 28))

    def _apply_so8_rotation(self, x: torch.Tensor) -> torch.Tensor:
        """SO(8)回転を適用"""
        batch_size, seq_len, d_model = x.shape

        # 回転行列を作成
        rotation_matrices = []
        for i in range(self.n_heads):
            rot_matrix = self.so8_group.exponential_map(self.rotation_params[i:i+1])
            extended_rot = self.so8_group.extend_generator_to_dimension(rot_matrix.squeeze(0), d_model)
            rotation_matrices.append(extended_rot)

        rotation_matrix = torch.stack(rotation_matrices, dim=0)

        # バッチとシーケンス次元で適用
        x_reshaped = x.view(batch_size * seq_len, d_model)
        rotated = torch.matmul(x_reshaped, rotation_matrix.transpose(-2, -1))
        return rotated.view(batch_size, seq_len, self.n_heads, d_model // self.n_heads)

    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        batch_size, seq_len, _ = x.shape

        # 線形変換
        q = self.q_proj(x).view(batch_size, seq_len, self.n_heads, self.d_head)
        k = self.k_proj(x).view(batch_size, seq_len, self.n_heads, self.d_head)
        v = self.v_proj(x).view(batch_size, seq_len, self.n_heads, self.d_head)

        # SO(8)回転適用
        q = self._apply_so8_rotation(q)
        k = self._apply_so8_rotation(k)
        v = self._apply_so8_rotation(v)

        # アテンション計算
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_head)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context = torch.matmul(attn_weights, v)
        context = context.view(batch_size, seq_len, self.d_model)

        return self.out_proj(context)

class SO8FeedForward(nn.Module):
    """SO(8)回転付きフィードフォワード"""

    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.linear1 = Linear(d_model, d_ff)
        self.linear2 = Linear(d_ff, d_model)
        self.dropout = Dropout(dropout)
        self.so8_group = SO8Group()

        # SO(8)回転パラメータ
        self.rotation_params = nn.Parameter(torch.randn(28))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        hidden = F.gelu(self.linear1(x))
        hidden = self.dropout(hidden)

        # SO(8)回転適用
        rotation_matrix = self.so8_group.exponential_map(self.rotation_params.unsqueeze(0))
        extended_rot = self.so8_group.extend_generator_to_dimension(rotation_matrix.squeeze(0), hidden.shape[-1])
        hidden = torch.matmul(hidden, extended_rot)

        return self.linear2(hidden)

class SO8TransformerBlock(nn.Module):
    """SO(8)トランスフォーマーブロック"""

    def __init__(self, d_model: int, n_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.attention = SO8Attention(d_model, n_heads, dropout)
        self.feed_forward = SO8FeedForward(d_model, d_ff, dropout)
        self.norm1 = LayerNorm(d_model)
        self.norm2 = LayerNorm(d_model)
        self.dropout = Dropout(dropout)

    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        # 残差接続 + 層正規化 + アテンション
        attn_out = self.attention(self.norm1(x), mask)
        x = x + self.dropout(attn_out)

        # 残差接続 + 層正規化 + フィードフォワード
        ff_out = self.feed_forward(self.norm2(x))
        x = x + self.dropout(ff_out)

        return x

class SO8Transformer(nn.Module):
    """SO(8)トランスフォーマーモデル"""

    def __init__(self, vocab_size: int, d_model: int = 512, n_heads: int = 8,
                 n_layers: int = 40, d_ff: int = 2048, dropout: float = 0.1,
                 max_seq_len: int = 1024):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.max_seq_len = max_seq_len

        # トークン埋め込みと位置埋め込み
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = self._create_so8_position_embedding(max_seq_len, d_model)

        # SO(8)トランスフォーマーブロック (40層)
        self.layers = nn.ModuleList([
            SO8TransformerBlock(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])

        # 出力層
        self.norm = LayerNorm(d_model)
        self.lm_head = Linear(d_model, vocab_size)

        # 重みの初期化
        self.apply(self._init_weights)

    def _create_so8_position_embedding(self, max_seq_len: int, d_model: int) -> nn.Embedding:
        """SO(8)群ベースの位置埋め込みを作成"""
        position_embeddings = torch.zeros(max_seq_len, d_model)
        so8_group = SO8Group()

        for pos in range(max_seq_len):
            # 位置に基づくSO(8)回転パラメータ
            params = torch.randn(28) * 0.1
            rotation_matrix = so8_group.exponential_map(params.unsqueeze(0))

            # 回転行列をベクトルに変換
            embedding = rotation_matrix.squeeze(0).flatten()[:d_model]
            if embedding.shape[0] < d_model:
                padding = torch.zeros(d_model - embedding.shape[0])
                embedding = torch.cat([embedding, padding])

            position_embeddings[pos] = embedding

        return nn.Embedding.from_pretrained(position_embeddings, freeze=False)

    def _init_weights(self, module):
        """重みの初期化"""
        if isinstance(module, Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, input_ids: torch.Tensor, labels: Optional[torch.Tensor] = None) -> dict:
        batch_size, seq_len = input_ids.shape

        # 埋め込み
        token_emb = self.token_embedding(input_ids)
        pos_emb = self.position_embedding(torch.arange(seq_len, device=input_ids.device))
        x = token_emb + pos_emb.unsqueeze(0)

        # 因果マスク作成
        causal_mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()
        causal_mask = causal_mask.to(input_ids.device)

        # SO(8)トランスフォーマーブロック適用
        for layer in self.layers:
            x = layer(x, causal_mask)

        x = self.norm(x)
        logits = self.lm_head(x)

        loss = None
        if labels is not None:
            loss = F.cross_entropy(
                logits.view(-1, self.vocab_size),
                labels.view(-1),
                ignore_index=-100
            )

        return {"logits": logits, "loss": loss}

def create_so8t_model(vocab_size: int, n_layers: int = 40, **kwargs) -> SO8Transformer:
    """
    SO(8)トランスフォーマーモデルを作成する関数
    デフォルトで40層を使用
    """
    logger.info(f"SO(8)モデル作成: vocab_size={vocab_size}, n_layers={n_layers}")
    model = SO8Transformer(
        vocab_size=vocab_size,
        n_layers=n_layers,
        **kwargs
    )
    logger.info("SO(8)モデル作成完了")
    return model


class SO8T_Colab_Trainer:
    """SO(8)トランスフォーマーのColabトレーニングクラス"""

    def __init__(self):
        # Colab環境向け設定
        self.d_model = 512
        self.n_heads = 8
        self.n_layers = 40  # 40層SO(8)トランスフォーマー
        self.d_ff = 2048
        self.dropout = 0.1
        self.max_seq_len = 512  # Colabメモリ制限考慮

        self.tokenizer = None
        self.model = None
        self.dataset = None

        # Colab用出力ディレクトリ
        self.output_dir = "/content/drive/MyDrive/so8t_nsfw_training"
        self.final_model_dir = "/content/drive/MyDrive/so8t_nsfw_final"

        os.makedirs(self.output_dir, exist_ok=True)
        os.makedirs(self.final_model_dir, exist_ok=True)

    def load_tokenizer(self):
        """トークナイザーを読み込み"""
        logger.info("トークナイザー読み込み中...")

        try:
            self.tokenizer = AutoTokenizer.from_pretrained("gpt2")
            if self.tokenizer.pad_token is None:
                self.tokenizer.pad_token = self.tokenizer.eos_token
        except Exception as e:
            logger.error(f"トークナイザー読み込みエラー: {e}")
            raise

        self.vocab_size = len(self.tokenizer)
        logger.info(f"トークナイザー読み込み完了: 語彙サイズ={self.vocab_size}")

    def create_model(self):
        """SO(8)モデルを作成"""
        logger.info("SO(8)モデルを作成中...")

        try:
            self.model = create_so8t_model(
                vocab_size=self.vocab_size,
                d_model=self.d_model,
                n_heads=self.n_heads,
                n_layers=self.n_layers,
                d_ff=self.d_ff,
                dropout=self.dropout,
                max_seq_len=self.max_seq_len
            )
            logger.info(f"モデル作成完了: {self.model.__class__.__name__}")
        except Exception as e:
            logger.error(f"モデル作成エラー: {e}")
            raise

    def load_dataset(self):
        """データセットを読み込み"""
        logger.info("データセット読み込み中...")

        data = []
        dataset_file = "datasets/gangimari_oho_full_dataset.jsonl"

        with open(dataset_file, 'r', encoding='utf-8') as f:
            for line in tqdm(f, desc="データ読み込み"):
                line = line.strip()
                if line:
                    try:
                        item = json.loads(line)
                        instruction = item.get('instruction', '')
                        output = item.get('output', '')
                        text = f"{instruction}\n{output}"

                        if len(text) > 10 and len(text) < self.max_seq_len * 2:
                            data.append({"text": text})
                    except json.JSONDecodeError:
                        continue

        self.dataset = Dataset.from_list(data)
        logger.info(f"データセット読み込み完了: {len(self.dataset)}サンプル")

    def tokenize_function(self, examples):
        """トークナイズ関数"""
        return self.tokenizer(
            examples["text"],
            truncation=True,
            padding="max_length",
            max_length=self.max_seq_len
        )

    def prepare_dataset(self):
        """データセットを準備"""
        logger.info("データセット準備中...")

        self.tokenized_dataset = self.dataset.map(
            self.tokenize_function,
            batched=True,
            remove_columns=["text"],
            desc="トークナイズ"
        )

        # labelsを作成
        self.tokenized_dataset = self.tokenized_dataset.map(
            lambda x: {"labels": x["input_ids"]},
            desc="ラベル作成"
        )

        logger.info("データセット準備完了")

    def setup_training_args(self):
        """トレーニング引数を設定"""
        logger.info("トレーニング引数設定中...")

        self.training_args = TrainingArguments(
            output_dir=self.output_dir,
            num_train_epochs=3,  # Colab用に短く
            per_device_train_batch_size=2,  # Colabメモリ制限
            gradient_accumulation_steps=8,
            learning_rate=2e-4,
            weight_decay=0.01,
            warmup_steps=100,
            logging_steps=10,
            save_steps=500,
            save_total_limit=3,
            evaluation_strategy="no",  # Colab用に評価なし
            fp16=True,  # Colab GPU対応
            dataloader_num_workers=0,  # Colab安定性
            report_to="none"
        )

        logger.info("トレーニング引数設定完了")

    def train(self):
        """トレーニング実行"""
        logger.info("トレーニング開始...")

        data_collator = DataCollatorForLanguageModeling(
            tokenizer=self.tokenizer,
            mlm=False
        )

        trainer = Trainer(
            model=self.model,
            args=self.training_args,
            train_dataset=self.tokenized_dataset,
            data_collator=data_collator,
        )

        trainer.train()

        # 最終モデル保存
        final_model_path = os.path.join(self.final_model_dir, "final_model")
        self.model.save_pretrained(final_model_path)
        self.tokenizer.save_pretrained(final_model_path)

        logger.info(f"トレーニング完了: {final_model_path}")
        return final_model_path

    def run_training(self):
        """トレーニング全体を実行"""
        try:
            self.load_tokenizer()
            self.create_model()
            self.load_dataset()
            self.prepare_dataset()
            self.setup_training_args()
            final_model_path = self.train()

            print(f"\n🎉 学習完了！モデル保存場所: {final_model_path}")
            return final_model_path

        except Exception as e:
            logger.error(f"学習中にエラーが発生しました: {e}")
            raise

# トレーニング実行
if __name__ == "__main__":
    trainer = SO8T_Colab_Trainer()
    final_model_path = trainer.run_training()
    print(f"\n🎉 学習完了！モデル保存場所: {final_model_path}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
データ読み込み: 1000000it [00:10, 93428.70it/s]


トークナイズ:   0%|          | 0/1000000 [00:00<?, ? examples/s]

ラベル作成:   0%|          | 0/1000000 [00:00<?, ? examples/s]

## 5. 使用方法と注意事項

### 使用方法:
1. 上記のセルを順番に実行してください
2. 最初のセルで環境準備
3. 2番目のセルでSO(8)モデル実装
4. 3番目のセルでNSFWデータセット生成
5. 4番目のセルでトレーニング実行

### 注意事項:
- このモデルは完全に架空のNSFWコンテンツ専用です
- 現実の薬物使用を推奨・描写しません
- 個人使用目的のみを想定しています
- Colabの無料枠では40層の学習は難しい場合があります
- 有料版Colab Pro+をおすすめします

### 推奨スペック:
- **GPU**: A100またはV100
- **RAM**: 32GB以上
- **ディスク**: 100GB以上

**⚠️ 倫理的注意**: このモデルは研究・教育目的のみ使用してください。**